# D00 — Pilot Forensic Reconstruction (RQ-D1)

| 항목 | 값 |
|---|---|
| research_question | E001 파일럿은 정확히 어느 지점에서 무엇을 잃었는가 |
| ticket_id | (없음 — Research Director 직접 지시) |
| created_at | 2026-08-27T21:31:54+09:00 |
| input Git SHA (base) | `bc0b7a087faf2328cbafdfa9b40bd426c5080d7d` |
| D branch | `claude-d/research-sandbox-v21` |
| raw artifact manifest SHA | `f92ef2aa14efc9c957ec827d3a5936f3f79443616e5d5622ffef9f5f9dd8f188` (`research_d/INPUT_SNAPSHOT_v21.json`) |
| label SHA | 해당 없음 — D는 label을 생산하지도 사용하지도 않음 |
| Python env | `/home/sieg/projects-wsl/ProjectFinal/.venv` (3.12) |
| random seed | 해당 없음 — 전량 결정론적 집계 |
| input paths | `.agent_worktrees/claude_b_e001_worker_0{1..4}/artifacts/e001_w0*/evidence`, `.agent_worktrees/claude_b_analysis_current/artifacts/e001_real_marts` |
| data grain | observation dir / web_target_group / mart row |

**A/B/C의 보고문은 입력으로 쓰지 않는다.** 전량 raw evidence와 frozen mart에서 재계산한다.


## 0. 입력 동결 확인

스냅샷 해시가 위 표와 다르면 이 노트북의 결과는 무효다.

In [1]:
import hashlib, json, sys
from pathlib import Path

REPO = Path("/home/sieg/projects-wsl/ProjectFinal")
RD = REPO / ".agent_worktrees/claude_d_research/research/landing_accessibility/research_d"
sys.path.insert(0, str(RD / "tools"))

snap_path = RD / "INPUT_SNAPSHOT_v21.json"
snap_sha = hashlib.sha256(snap_path.read_bytes()).hexdigest()
snap = json.loads(snap_path.read_text())
print("input snapshot sha256:", snap_sha)
print("SSOT files frozen   :", len(snap["ssot_authority"]["files"]))
print("remote heads frozen :", len(snap["remote_heads"]))
print("evidence roots      :", {k: v.get("observation_count") for k, v in snap["evidence_roots"].items()})

input snapshot sha256: f92ef2aa14efc9c957ec827d3a5936f3f79443616e5d5622ffef9f5f9dd8f188
SSOT files frozen   : 11
remote heads frozen : 34
evidence roots      : {'w01': 15, 'w02': 20, 'w03': 17, 'w04': 14}


## 1. 독립 재구성 실행

`tools/rq_d1_reconstruct.py`는 순수 함수 집합이다. hidden state가 없다.

In [2]:
import rq_d1_reconstruct as rq

rows = rq.scan_evidence()
repeats = rq.classify_repeats(rows)
mart = rq.load_mart()

print(f"observation dirs        : {len(rows)}")
print(f"distinct web_target_grp : {len({r['wtg'] for r in rows})}")
print(f"repeat targets          : {len(repeats)}")
print(f"empty dirs              : {sum(1 for r in rows if r['artifact_count'] == 0)}")
print(f"total raw bytes         : {sum(r['artifact_bytes'] for r in rows):,}")

observation dirs        : 66
distinct web_target_grp : 59
repeat targets          : 7
empty dirs              : 6
total raw bytes         : 753,676,839


## 2. F1 — 반복 실행 7건은 두 개의 다른 기계적 지문으로 갈린다

In [3]:
for wtg, v in sorted(repeats.items(), key=lambda kv: kv[1]["gap_seconds"]):
    print(f"{wtg}  {'/'.join(v['workers']):>4}  gap={v['gap_seconds']:>7.3f}s  "
          f"{v['gap_class']:<5}  artifacts={v['artifact_counts']}  {v['classification']}")

gaps = sorted(v["gap_seconds"] for v in repeats.values())
print()
print("gap 분포:", [round(g, 2) for g in gaps])
print("두 군집 사이 빈 구간:", f"{max(g for g in gaps if g < 20):.2f}s ~ {min(g for g in gaps if g >= 20):.2f}s")

9390ef32addf32bf   w02  gap=  6.250s  SHORT  artifacts=[15, 15]  DUPLICATE_LAUNCH_BOTH_COMPLETE
e1fadb214cde51c0   w02  gap=  6.282s  SHORT  artifacts=[12, 12]  DUPLICATE_LAUNCH_BOTH_COMPLETE
b728911c9782edb8   w02  gap=  6.949s  SHORT  artifacts=[33, 33]  DUPLICATE_LAUNCH_BOTH_COMPLETE
13ed070478ef62c3   w02  gap=  7.572s  SHORT  artifacts=[6, 6]  DUPLICATE_LAUNCH_BOTH_COMPLETE
2cd43b99c1ed87cf   w03  gap= 46.494s  LONG   artifacts=[0, 0]  RETRY_BOTH_EMPTY
ff3ee504792f6cfc   w02  gap= 46.588s  LONG   artifacts=[0, 0]  RETRY_BOTH_EMPTY
dd5061eb74e2d4d4   w03  gap= 46.662s  LONG   artifacts=[0, 0]  RETRY_BOTH_EMPTY

gap 분포: [6.25, 6.28, 6.95, 7.57, 46.49, 46.59, 46.66]
두 군집 사이 빈 구간: 7.57s ~ 46.49s


### 판정 (OBSERVATION)

- **SHORT 4건** (6.25~7.57s, 전부 w02): 양쪽 모두 sealed, 쌍마다 artifact 수 동일
  → 첫 시도가 **성공한 뒤** 재실행. exactly-once 위반.
- **LONG 3건** (46.49~46.66s): 양쪽 모두 빈 디렉터리 → timeout 후 재시도, 재시도도 실패.

46.5s ± 0.09s의 응집은 고정 timeout의 지문이다.

## 3. F4/F5 — 분모가 조용히 줄어드는 두 지점

In [4]:
import re
raw_wtg = {r["wtg"] for r in rows}
empty_wtg = {w for w in raw_wtg if all(r["artifact_count"] == 0 for r in rows if r["wtg"] == w)}
mart_wtg = {re.sub(r"^e001_full-wtg_([0-9a-f]+)-.*", r"\1", r["evidence_run_id"])
            for r in mart["landing"]}
land_t = {r["web_target_id"] for r in mart["landing"]}
task_t = {r["web_target_id"] for r in mart["task"]}

print(f"observation dirs              : {len(rows)}")
print(f"distinct targets attempted    : {len(raw_wtg)}")
print(f"targets with any evidence     : {len(raw_wtg - empty_wtg)}")
print(f"targets in landing mart       : {len(land_t)}")
print(f"targets in task mart          : {len(task_t)}")
print()
print("mart에서 사라진 target :", sorted(raw_wtg - mart_wtg))
print("빈 디렉터리 target     :", sorted(empty_wtg))
print("두 집합이 동일한가     :", (raw_wtg - mart_wtg) == empty_wtg)
print()
print(f"landing에 있으나 task 행이 없는 target: {len(land_t - task_t)}")

observation dirs              : 66
distinct targets attempted    : 59
targets with any evidence     : 56
targets in landing mart       : 56
targets in task mart          : 31

mart에서 사라진 target : ['2cd43b99c1ed87cf', 'dd5061eb74e2d4d4', 'ff3ee504792f6cfc']
빈 디렉터리 target     : ['2cd43b99c1ed87cf', 'dd5061eb74e2d4d4', 'ff3ee504792f6cfc']
두 집합이 동일한가     : True

landing에 있으나 task 행이 없는 target: 25


### 판정 (ANALYSIS)

1. mart에 없는 3 target = 재시도까지 실패한 3 target과 **정확히 동일 집합**.
   `NO_EVIDENCE` 행조차 남지 않아 mart만 읽는 downstream은 손실을 볼 수 없다.
   사라진 것이 "46초 timeout을 두 번 유발한 사이트"이므로 **informative missingness**다.
2. landing 56 → task 31의 25 target 손실(guard block)도 같은 방식으로 조용하다.

## 4. F6~F9 — 축별 가용성과 archetype 편중

In [5]:
from collections import Counter
import statistics as st

te = mart["task"]
print("Axis A criterion rows :", len(mart["criterion"]))
print("Axis B task rows      :", len(te))
for f in ("NED", "IED", "MPFED"):
    print(f"  {f} non-null        : {sum(1 for r in te if r[f] is not None)}/{len(te)}")
print("  endpoint_status     :", dict(Counter(r["endpoint_status"] for r in te)))
print()
cov = [r["max_overlay_coverage"] for r in mart["landing"] if r["max_overlay_coverage"] is not None]
print(f"Axis C overlay coverage: {len(cov)}/{len(mart['landing'])} non-null, "
      f"median={st.median(cov):.4f}, max={max(cov)}")
print("Axis C interrupt rows  :", len(mart["interrupt"]))
print()
FROZEN = ["QUERY", "CONTENT_OPEN", "ITEM_DETAIL", "PLACE_LOOKUP",
          "COMMUNICATION_ENTRY", "FINANCIAL_ACTION_ENTRY", "UTILITY_ENTRY"]
arch = Counter(r["interaction_archetype"] for r in te)
for a in FROZEN:
    n = arch.get(a, 0)
    rule = "정상" if n >= 5 else ("LOW_N" if n >= 3 else ("과해석 금지" if n >= 1 else "부재"))
    print(f"  {a:<24} n={n:<3} {rule}")

Axis A criterion rows : 0
Axis B task rows      : 31
  NED non-null        : 0/31
  IED non-null        : 0/31
  MPFED non-null        : 0/31
  endpoint_status     : {'AUTH_GATE_REACHED': 11, 'UNRESOLVED': 18, 'CAPTCHA': 1, 'PAYMENT_GATE_REACHED': 1}

Axis C overlay coverage: 56/56 non-null, median=0.1281, max=1.0
Axis C interrupt rows  : 235

  QUERY                    n=0   부재
  CONTENT_OPEN             n=3   LOW_N
  ITEM_DETAIL              n=16  정상
  PLACE_LOOKUP             n=4   LOW_N
  COMMUNICATION_ENTRY      n=2   과해석 금지
  FINANCIAL_ACTION_ENTRY   n=4   LOW_N
  UTILITY_ENTRY            n=2   과해석 금지


### 판정 (OBSERVATION)

- **Axis A**: criterion 행 0 — byte 수준 사실.
- **Axis B**: NED조차 0/31. SSOT 00 §8.4의 partial NED 보존이 미구현이다.
  `AUTH_GATE_REACHED` 11건은 경로를 실제로 전진했는데도 NED가 비어 있다.
  → "MPFED 0/59"를 detector 결함 하나로 설명하면 부족하다.
- **Axis C**: 56/56 가용. 단 primary_action_occlusion은 task binding 없이 계산돼
  **page-level로만 유효**하다.
- **archetype**: QUERY n=0, ITEM_DETAIL 16/31(51.6%) 지배.
  ExcessDepth의 same-archetype median이 6개 archetype 중 5개에서 n≤4다.

## 5. 결과 export

In [6]:
out = RD / "results" / "RQ_D1_reconstruction.json"
print("canonical result:", out)
print("bytes:", out.stat().st_size)
print()
print("verdict: SUPPORTED (재구성 성공) + 신규 P1 finding 2건 (F4 조용한 분모손실, F5 guard 25건 무기록)")
print("상세 서술: results/RQ_D1_FINDINGS.md")

canonical result: /home/sieg/projects-wsl/ProjectFinal/.agent_worktrees/claude_d_research/research/landing_accessibility/research_d/results/RQ_D1_reconstruction.json
bytes: 6415

verdict: SUPPORTED (재구성 성공) + 신규 P1 finding 2건 (F4 조용한 분모손실, F5 guard 25건 무기록)
상세 서술: results/RQ_D1_FINDINGS.md
